# Question 12: Implement KV Cache for Autoregressive Generation (SOLUTION)

### Difficulty: Medium

### Problem Statement

During autoregressive text generation, a transformer model generates one token at a time. At each step, it recomputes attention over **all** previous tokens -- even though their key-value pairs have not changed. This is wasteful.

A **KV Cache** stores the key and value projections from all previous timesteps so that at each new step, only the **new token's** query, key, and value need to be computed. The cached keys and values are simply concatenated with the new ones before computing attention.

Generation has two phases: **prefill** runs the whole prompt through the model in one call (filling the cache), then **decode** feeds one new token per step. Per decode step this turns attention over the context from recomputing all t positions (O(t²·d) for the scores) into one query against t cached keys (O(t·d)), at the cost of memory: 2 × layers × batch × heads × seq_len × d_head values.

Your task is to:
1. Implement a `KVCache` class that stores and updates past key-value tensors.
2. Implement a `CachedAttention` module that uses the KV cache during generation.
3. Show that the cached version produces **identical** outputs to full recomputation.
4. Benchmark the speed difference.

---

### Requirements

1. **`KVCache` class**
   - `update(new_k, new_v)`: Append new key/value tensors to the cache and return the full (cached + new) key/value tensors.
   - `get()`: Return the current cached key and value tensors.
   - `reset()`: Clear the cache.

2. **`CachedAttention(nn.Module)`**
   - A single-head (or multi-head) attention layer with Q, K, V projections.
   - In `forward()`, accept an optional `KVCache`. When provided, only compute Q/K/V for the **new** token(s), append K/V to the cache, and attend over cached + new keys.
   - The new tokens may be a single decode token **or a multi-token chunk** (prefill of the prompt into an empty cache, or a chunk appended to a non-empty one). A chunk still needs a causal mask among its own tokens, offset by the cached length: new token `i` sits at absolute position `past_len + i` and may attend to keys `0 .. past_len + i`. A single decode token may attend to everything in the cache, so it needs no mask.

3. **Validation**
   - Generate a sequence token-by-token using KV cache.
   - Prefill the first part of the sequence in one cached call, then decode the rest token-by-token.
   - Compute the same sequence in one full (causally masked) forward pass.
   - Verify the outputs match with `torch.allclose()`.

4. **Benchmarking**
   - Time both approaches over a longer sequence and show the speedup.

---

### Constraints

- Use only PyTorch operations.
- Use a causal (lower-triangular) attention mask for the non-cached version.
- The cached version should produce the same outputs as the non-cached version (up to floating-point error — the reductions run in a different order, so compare with `torch.allclose`).

---

### Company Tags

Anthropic, OpenAI, Meta, Perplexity, Together AI

---

<details>
  <summary>Hint</summary>

  - The KV cache stores tensors of shape `(batch_size, num_heads, seq_len, d_head)`. Each `update()` call concatenates along the `seq_len` dimension (dim=2).
  - During cached generation, the query tensor has `seq_len=1` (just the new token), but keys and values have the full sequence length from the cache.
  - The attention output shape is `(batch_size, num_heads, 1, d_head)` -- just the output for the new token.
  - Make sure the causal mask prevents attending to future tokens — in full-recomputation mode and for a multi-token chunk passed with a cache (`torch.ones(S, past_len + S).tril(diagonal=past_len)`).
  - With positional encodings (learned, or RoPE applied to q and k), the new token's position is `past_len`, not 0 — cached keys keep the positions they were computed with.
  - Production caches preallocate `(batch, heads, max_seq_len, d_head)` buffers and write in place (or use pages, as in vLLM) instead of `torch.cat`, which copies the whole cache every step.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import math

In [ ]:
# Configuration
torch.manual_seed(42)

batch_size = 2
d_model = 64
num_heads = 4
d_head = d_model // num_heads
max_seq_len = 32

device = "cpu"
print(f"Config: batch_size={batch_size}, d_model={d_model}, num_heads={num_heads}, d_head={d_head}")

In [ ]:
class KVCache:
    """
    Stores past key and value tensors for autoregressive generation.
    
    The cache stores tensors of shape (batch_size, num_heads, seq_len, d_head).
    Each call to update() appends new K/V along the seq_len dimension.
    """
    
    def __init__(self):
        self.k = None
        self.v = None
    
    def update(self, new_k, new_v):
        """
        Append new key/value tensors to the cache.
        
        Args:
            new_k: (batch_size, num_heads, new_seq_len, d_head)
            new_v: (batch_size, num_heads, new_seq_len, d_head)
            
        Returns:
            full_k, full_v: The concatenated (cached + new) key and value tensors
        """
        if self.k is None:
            self.k = new_k
            self.v = new_v
        else:
            self.k = torch.cat([self.k, new_k], dim=2)
            self.v = torch.cat([self.v, new_v], dim=2)
        return self.k, self.v
    
    def get(self):
        """Return the current cached (k, v) tensors."""
        return self.k, self.v
    
    def reset(self):
        """Clear the cache."""
        self.k = None
        self.v = None


class CachedAttention(nn.Module):
    """
    Multi-head attention with optional KV cache support.
    
    During full forward pass: computes Q, K, V for all positions, applies causal mask.
    During cached generation: computes Q, K, V only for the new token(s), uses cache for past K/V.
    """
    
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x, kv_cache=None):
        """
        Args:
            x: Input tensor.
               - Full mode: (batch_size, seq_len, d_model)
               - Cached mode: (batch_size, new_len, d_model) -- the new token(s): one token
                 per decode step, or a whole prompt chunk (prefill)
            kv_cache: Optional KVCache instance. If provided, use cached generation.
            
        Returns:
            output: Attention output, same shape as x.
        """
        B, S, _ = x.shape
        
        # Step 1: Project to Q, K, V
        q = self.W_q(x)  # (B, S, d_model)
        k = self.W_k(x)
        v = self.W_v(x)
        
        # Step 2: Reshape to (B, num_heads, S, d_head)
        q = q.view(B, S, self.num_heads, self.d_head).transpose(1, 2)
        k = k.view(B, S, self.num_heads, self.d_head).transpose(1, 2)
        v = v.view(B, S, self.num_heads, self.d_head).transpose(1, 2)
        
        # Step 3: Handle KV cache
        if kv_cache is not None:
            # Update cache and get full K, V (past + current)
            k, v = kv_cache.update(k, v)
            # q has shape (B, num_heads, S, d_head) -- only the new token(s)
            # k, v have shape (B, num_heads, past_len + S, d_head)
        
        # Step 4: Compute scaled dot-product attention
        # scores: (B, num_heads, q_len, kv_len)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_head)
        
        # Step 5: Apply causal mask whenever more than one query position is processed --
        # full recomputation AND a multi-token chunk (prefill) passed with a cache.
        # New token i sits at absolute position past_len + i and may see keys 0..past_len + i.
        # A single decode token (S == 1) may attend to everything in the cache: no mask needed.
        if S > 1:
            kv_len = k.shape[2]
            past_len = kv_len - S
            causal_mask = torch.ones(S, kv_len, dtype=torch.bool, device=x.device).tril(diagonal=past_len)
            scores = scores.masked_fill(~causal_mask, float('-inf'))
        
        # Step 6: Softmax and weighted sum
        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, v)  # (B, num_heads, q_len, d_head)
        
        # Step 7: Reshape back to (B, q_len, d_model)
        q_len = q.shape[2]
        output = output.transpose(1, 2).contiguous().view(B, q_len, self.d_model)
        
        # Step 8: Output projection
        return self.W_o(output)


print("Classes defined successfully.")

In [ ]:
# ============================================================
# Validation: Compare cached vs. full-recomputation outputs
# ============================================================

torch.manual_seed(42)
attn = CachedAttention(d_model, num_heads).to(device)
attn.eval()

# Create a random input sequence
seq_len = 16
x_full = torch.randn(batch_size, seq_len, d_model, device=device)

# --- Method 1: Full recomputation (no cache) ---
# Process the entire sequence at once with causal masking
with torch.no_grad():
    full_output = attn(x_full, kv_cache=None)
print(f"Full output shape: {full_output.shape}")

# --- Method 2: Token-by-token with KV cache ---
cache = KVCache()
cached_outputs = []
with torch.no_grad():
    for t in range(seq_len):
        x_t = x_full[:, t:t+1, :]  # (batch_size, 1, d_model)
        out_t = attn(x_t, kv_cache=cache)  # (batch_size, 1, d_model)
        cached_outputs.append(out_t)

cached_output = torch.cat(cached_outputs, dim=1)  # (batch_size, seq_len, d_model)
print(f"Cached output shape: {cached_output.shape}")

# --- Method 3: Prefill a prompt chunk into the cache, then decode token-by-token ---
prompt_len = 6
cache = KVCache()
with torch.no_grad():
    chunk_outputs = [attn(x_full[:, :prompt_len, :], kv_cache=cache)]  # (batch_size, prompt_len, d_model)
    for t in range(prompt_len, seq_len):
        chunk_outputs.append(attn(x_full[:, t:t+1, :], kv_cache=cache))
prefill_output = torch.cat(chunk_outputs, dim=1)

# --- Compare ---
for name, out in [("token-by-token", cached_output), ("prefill + decode", prefill_output)]:
    match = torch.allclose(full_output, out, atol=1e-5)
    max_diff = (full_output - out).abs().max().item()
    print(f"\n{name}: outputs match: {match}, max absolute difference: {max_diff:.2e}")
    assert match, f"{name} does not match full recomputation! Max diff: {max_diff:.2e}"
print("\nValidation PASSED: KV cache reproduces full recomputation (token-by-token and prefill + decode).")

In [ ]:
# ============================================================
# Benchmarking: Speed comparison
# ============================================================

benchmark_seq_len = 128
num_runs = 5

x_bench = torch.randn(batch_size, benchmark_seq_len, d_model, device=device)

# --- Benchmark full recomputation (token-by-token, no cache) ---
times_no_cache = []
for _ in range(num_runs):
    start = time.time()
    with torch.no_grad():
        for t in range(benchmark_seq_len):
            # Without cache: recompute attention over all tokens up to t
            _ = attn(x_bench[:, :t+1, :], kv_cache=None)
    times_no_cache.append(time.time() - start)

# --- Benchmark with KV cache ---
times_with_cache = []
for _ in range(num_runs):
    cache = KVCache()
    start = time.time()
    with torch.no_grad():
        for t in range(benchmark_seq_len):
            _ = attn(x_bench[:, t:t+1, :], kv_cache=cache)
    times_with_cache.append(time.time() - start)

avg_no_cache = sum(times_no_cache) / num_runs
avg_with_cache = sum(times_with_cache) / num_runs
speedup = avg_no_cache / avg_with_cache

print(f"Sequence length: {benchmark_seq_len}")
print(f"Avg time WITHOUT cache: {avg_no_cache:.4f}s")
print(f"Avg time WITH cache:    {avg_with_cache:.4f}s")
print(f"Speedup: {speedup:.2f}x")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("kv-cache", KVCache=KVCache, CachedAttention=CachedAttention)